# Scenario B1 Mixed: Multiple Suppliers — No REC (Mixed Prosumers)

**Description:** Two independent suppliers, no REC, with mixed prosumer profiles.
Prosumers are modelled as **full prosumers** — both local load AND RES generation
are billed, reflecting realistic household-with-PV behaviour.
Each balancing group contains a single member type (consumers only or prosumers only).

**Participant–supplier assignment:**
| Supplier / BG | Participants |
|---|---|
| SUP_A / BG_A | consumer_001–006 (6 consumers) |
| SUP_B / BG_B | prosumer_001–003 (3 prosumers) |

**Configuration:**
| Parameter | Value |
|---|---|
| Suppliers | 2 — SUP_A, SUP_B |
| Balancing Groups | 2 — BG_A (6 consumers), BG_B (3 prosumers) |
| RECs | None |
| Prosumer type | **Mixed** (RES + local load both billed) |
| Battery | None |

**Research role:** Multi-supplier baseline with homogeneous balancing groups.
BG_A is a pure-consumer group (no generation), while BG_B carries all PV
generation alongside prosumer loads. Compared to A1-mixed, reveals how
multi-supplier fragmentation with member-type separation reshapes each
supplier's balancing position and P&L.

**Comparison pairs (mixed-prosumer track):**
- B1-mixed vs A1-mixed → effect of multi-supplier competition (no REC)
- B1-mixed vs B2-mixed → incremental value of adding a cross-supplier REC
- B1-mixed vs B2-forecasts-mixed → REC value under explicit forecast scheduling

## 1. Import Dependencies
Import the EnergyMarketOperations class which handles the complete market simulation pipeline.

In [ ]:
from energy_market_operations import EnergyMarketOperations

## 2. Initialize Pipeline
Load the scenario configuration from JSON and create the pipeline instance. The config specifies participants, suppliers, market prices, and network topology.

In [ ]:
CONFIG_FILE = "B1_multiple_suppliers_without_rec(mixed prosumers).json"
pipe = EnergyMarketOperations(CONFIG_FILE, scenario_name="B1_mixed_prosumers")

## 3. Run Full Pipeline
Execute the complete market simulation:
1. **Day-Ahead Market** - Schedule energy positions based on DA forecasts (per BG)
2. **Intra-Day Market** - Adjust positions with updated ID forecasts (per BG)
3. **REC Settlement** - (Skipped for B1 - no REC configured)
4. **Balancing Market** - Settle imbalances at dual pricing (per BG)
5. **Supplier Billing** - Calculate final costs per participant

In [ ]:
pipe.run_all()

## 4. Financial Summary
Display aggregated financial results including total revenues, costs, and profit/loss for each supplier.

In [ ]:
pipe.summary()

## 5. Financial Visualization
Plot breakdown of revenues and costs by category (DA purchases, ID adjustments, balancing costs, customer sales).

In [ ]:
pipe.plot_financials()

## 6. Imbalance Analysis
Visualize system imbalances over time showing the difference between scheduled positions and actual metered values.

In [ ]:
pipe.plot_imbalances()

## 7. Supplier Comparison
Compare financial performance across the two suppliers (SUP_A, SUP_B).

In [ ]:
pipe.es_monthly_analysis_df.groupby('supplier_id').agg({
    'profit_loss_eur': 'sum',
    'imbalance_mwh': 'sum'
})

In [ ]:
import pandas as pd, numpy as np
df = pipe.es_monthly_analysis_df.copy()
df['datetime_dt'] = pd.to_datetime(df['datetime'])
sid = 'SUP_A'
sd = df[df['supplier_id'] == sid].sort_values('datetime_dt')
bg_id = sd['balancing_group_id'].iloc[0]
print(f"SUPPLIER: {sid} / {bg_id}")
bg_actual = sd['balancing_group_actual_mwh'].sum()
bg_forecast = sd['balancing_group_forecast_mwh'].sum()
imbalance = sd['imbalance_mwh'].sum()
bal_reward = sd['revenue_balancing_rewards_eur'].sum()
bal_penalty = sd['cost_balancing_penalties_eur'].sum()
print(f"BG Actual: {bg_actual:.4f}, Forecast: {bg_forecast:.4f}, Imb: {imbalance:+.4f} ({'SHORT' if imbalance>0 else 'LONG'})")
print(f"Bal Reward: €{bal_reward:.2f}, Penalty: €{bal_penalty:.2f}, Net: €{bal_reward-bal_penalty:+.2f}")
rev_rs = sd['revenue_retail_sales_eur'].sum()
rev_ms = sd['revenue_energy_market_sales_eur'].sum()
rev_br = bal_reward
total_rev = sd['total_revenue_eur'].sum()
print(f"Revenue: RS=€{rev_rs:.2f}({rev_rs/total_rev*100:.2f}%), MS=€{rev_ms:.2f}({rev_ms/total_rev*100:.2f}%), BR=€{rev_br:.2f}({rev_br/total_rev*100:.2f}%), Total=€{total_rev:.2f}")
cost_mp = sd['cost_energy_market_purchases_eur'].sum()
cost_bp = bal_penalty
cost_rp = sd['cost_retail_purchases_eur'].sum()
total_cost = sd['total_costs_eur'].sum()
print(f"Costs: MP=€{cost_mp:.2f}({cost_mp/total_cost*100:.2f}%), BP=€{cost_bp:.2f}({cost_bp/total_cost*100:.2f}%), RP=€{cost_rp:.2f}({cost_rp/total_cost*100:.2f}%), Total=€{total_cost:.2f}")
profit = sd['profit_loss_eur'].sum()
print(f"Profit: €{profit:.2f}, avg=€{sd['profit_loss_eur'].mean():.2f}, min=€{sd['profit_loss_eur'].min():.2f}, max=€{sd['profit_loss_eur'].max():.2f}")
print(f"Load: {sd['actual_load_mwh'].sum():.4f}, Gen: {sd['actual_gen_mwh'].sum():.4f}")

In [ ]:
sid = 'SUP_B'
sd = df[df['supplier_id'] == sid].sort_values('datetime_dt')
bg_id = sd['balancing_group_id'].iloc[0]
print(f"SUPPLIER: {sid} / {bg_id}")
bg_actual = sd['balancing_group_actual_mwh'].sum()
bg_forecast = sd['balancing_group_forecast_mwh'].sum()
imbalance = sd['imbalance_mwh'].sum()
bal_reward = sd['revenue_balancing_rewards_eur'].sum()
bal_penalty = sd['cost_balancing_penalties_eur'].sum()
print(f"BG Actual: {bg_actual:.4f}, Forecast: {bg_forecast:.4f}, Imb: {imbalance:+.4f} ({'SHORT' if imbalance>0 else 'LONG'})")
print(f"Bal Reward: €{bal_reward:.2f}, Penalty: €{bal_penalty:.2f}, Net: €{bal_reward-bal_penalty:+.2f}")
rev_rs = sd['revenue_retail_sales_eur'].sum()
rev_ms = sd['revenue_energy_market_sales_eur'].sum()
rev_br = bal_reward
total_rev = sd['total_revenue_eur'].sum()
print(f"Revenue: RS=€{rev_rs:.2f}({rev_rs/total_rev*100:.2f}%), MS=€{rev_ms:.2f}({rev_ms/total_rev*100:.2f}%), BR=€{rev_br:.2f}({rev_br/total_rev*100:.2f}%), Total=€{total_rev:.2f}")
cost_mp = sd['cost_energy_market_purchases_eur'].sum()
cost_bp = bal_penalty
cost_rp = sd['cost_retail_purchases_eur'].sum()
total_cost = sd['total_costs_eur'].sum()
print(f"Costs: MP=€{cost_mp:.2f}({cost_mp/total_cost*100:.2f}%), BP=€{cost_bp:.2f}({cost_bp/total_cost*100:.2f}%), RP=€{cost_rp:.2f}({cost_rp/total_cost*100:.2f}%), Total=€{total_cost:.2f}")
profit = sd['profit_loss_eur'].sum()
print(f"Profit: €{profit:.2f}, avg=€{sd['profit_loss_eur'].mean():.2f}, min=€{sd['profit_loss_eur'].min():.2f}, max=€{sd['profit_loss_eur'].max():.2f}")
print(f"Load: {sd['actual_load_mwh'].sum():.2f}, Gen: {sd['actual_gen_mwh'].sum():.2f}")

In [ ]:
# Monthly BG positions for both suppliers
for sid in ['SUP_A', 'SUP_B']:
    sd = df[df['supplier_id'] == sid].sort_values('datetime_dt')
    print(f"\n{sid} Monthly BG:")
    for _, r in sd.iterrows():
        print(f"  {r['datetime']}: act={r['balancing_group_actual_mwh']:.4f} fcast={r['balancing_group_forecast_mwh']:.4f} imb={r['imbalance_mwh']:+.4f}")

In [ ]:
# SUP_A Monthly Costs & Revenue
sd = df[df['supplier_id'] == 'SUP_A'].sort_values('datetime_dt')
print("SUP_A Monthly Costs:")
for _, r in sd.iterrows():
    print(f"  {r['datetime']}: MP=€{r['cost_energy_market_purchases_eur']:.2f} BP=€{r['cost_balancing_penalties_eur']:.2f} FI=€{r['cost_retail_purchases_eur']:.2f} Total=€{r['total_costs_eur']:.2f}")
print("\nSUP_A Monthly P/L:")
for _, r in sd.iterrows():
    print(f"  {r['datetime']}: P/L=€{r['profit_loss_eur']:.2f}")

In [ ]:
# SUP_B Monthly Costs & Revenue
sd = df[df['supplier_id'] == 'SUP_B'].sort_values('datetime_dt')
print("SUP_B Monthly Costs:")
for _, r in sd.iterrows():
    print(f"  {r['datetime']}: MP=€{r['cost_energy_market_purchases_eur']:.2f} BP=€{r['cost_balancing_penalties_eur']:.2f} FI=€{r['cost_retail_purchases_eur']:.2f} Total=€{r['total_costs_eur']:.2f}")
print("\nSUP_B Monthly P/L:")
for _, r in sd.iterrows():
    print(f"  {r['datetime']}: P/L=€{r['profit_loss_eur']:.2f}")

In [ ]:
# Config participant structure
cfg = pipe.config
# Check keys
if cfg.get('consumers'):
    print("Consumer keys:", list(cfg['consumers'][0].keys()))
if cfg.get('prosumers'):
    print("Prosumer keys:", list(cfg['prosumers'][0].keys()))

for s in cfg['suppliers']:
    sid = s['supplier_id']
    sname = s['supplier_name']
    bgs = [bg['balancing_group_id'] for bg in s['balancing_groups']]
    consumers = [c.get('participant_id', c.get('id', c.get('name', '?'))) for c in cfg.get('consumers', []) if c['supplier']['supplier_id'] == sid]
    prosumers = [p.get('participant_id', p.get('id', p.get('name', '?'))) for p in cfg.get('prosumers', []) if p['supplier']['supplier_id'] == sid]
    print(f"{sid} ({sname}): BGs={bgs}, #Consumers={len(consumers)}, #Prosumers={len(prosumers)}")

print(f"\nCombined Revenue: €{df['total_revenue_eur'].sum():.2f}")
print(f"Combined Costs:   €{df['total_costs_eur'].sum():.2f}")
print(f"Combined Profit:  €{df['profit_loss_eur'].sum():.2f}")
print(f"Total Load: {df['actual_load_mwh'].sum():.2f} MWh")
print(f"Total Gen:  {df['actual_gen_mwh'].sum():.2f} MWh")